In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from natsort import natsorted
import numpy as np
import re
import sys

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'H:\phd stuff\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]



In [2]:
dir = rf"./data/diffraction_monitor_data"
os.makedirs(dir, exist_ok=True)

In [3]:
try:
    data_path = f"{dir}/20260422_diffraction_n_3.4_ff_0.2172_ffh_0.225_schulz.h5"
    data_old = AM.read_hdf5_as_dict(data_path)
    data_old.keys()
except FileNotFoundError:
    print("File not found.")
    data_old = {}
except Exception as e:
    print(f"An error occurred: {e}")
    data_old = {}

folder_path = rf"H:\phd stuff\tidy3d\data\20260423 LSU Transmission n_3.4 ff_0.2172 ffh_0.225 Schulz Diffraction No FixedAngleSpec"

File not found.


In [4]:
reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "reference.txt"),get_ref=False)

amps_ref = reference_object.sim_data["diffraction"].amps
Pinc = np.abs(amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2

reference_exit = reference_object.sim_data["flux1"].flux


Configured successfully.


15:40:27 W. Europe Daylight Time Billed flex credit cost: 0.109.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
n_values = []
ff_values = []
size_values = []
z_values = []
values =data_old["transmission_data"] if "transmission_data" in data_old.keys() else {}
reference_entry=None
# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      try:
        z_value = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', dirpath).group(1))
      except AttributeError:
        print(f"Could not extract z_value from directory: {dirpath}")
        continue
      
      z_values.append(z_value)
      for filename in filenames:
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ffr_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            sample = float(re.search(r'sample_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            try:
                test_val = values[n_value][ff][z_value][size][sample]
                print(f"Data for n={n_value}, ff={ff}, z={z_value}, size={size}, sample={sample} already exists. Skipping file: {filename}")
            except KeyError:
                #Retrieve simulation data 
                if os.path.isfile(os.path.join(dirpath, filename)):
                  file=os.path.join(dirpath, filename)
                  structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
                  sim_data_i = structure_1.sim_data
                  transmission_entry = sim_data_i['flux2'].flux
                  transmission_exit = sim_data_i['flux1'].flux
                  if n_value not in values.keys():
                      values[n_value] = {}
                  if ff not in values[n_value].keys():
                      values[n_value][ff] = {}
                  if z_value not in values[n_value][ff].keys():
                        values[n_value][ff][z_value] = {}
                  if size not in values[n_value][ff][z_value].keys():
                        values[n_value][ff][z_value][size] = {}
                  if sample not in values[n_value][ff][z_value][size].keys():
                        values[n_value][ff][z_value][size][sample] = {}

                  #p = co and s = cross
                  amps = structure_1.sim_data["diffraction"].amps
                  T_co = abs(amps.sel(polarization="p"))**2/Pinc
                  T_cross = abs(amps.sel(polarization="s"))**2/Pinc
                  T_co_total = T_co.sum(dim=("orders_x", "orders_y"))
                  T_cross_total = T_cross.sum(dim=("orders_x", "orders_y"))
                  T_ballistic = np.abs(amps.sel(orders_x=0, orders_y=0, polarization="p")/amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2
                  lambdas = td.C_0 / structure_1.sim_data["diffraction"].f
                  T_total = transmission_exit/reference_exit

                  values[n_value][ff][z_value][size][sample]["T_ballistic"] = T_ballistic
                  values[n_value][ff][z_value][size][sample]["T_co"] = T_co_total
                  values[n_value][ff][z_value][size][sample]["T_cross"] = T_cross_total
                  values[n_value][ff][z_value][size][sample]["T_total"] = T_total

        except Exception as e:
            print("Error:", e)
            continue
       

# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)
z_unique = np.unique(z_values)




Could not extract z_value from directory: H:\phd stuff\tidy3d\data\20260423 LSU Transmission n_3.4 ff_0.2172 ffh_0.225 Schulz Diffraction No FixedAngleSpec
Could not extract z_value from directory: H:\phd stuff\tidy3d\data\20260423 LSU Transmission n_3.4 ff_0.2172 ffh_0.225 Schulz Diffraction No FixedAngleSpec\n_3.40
Configured successfully.


15:40:33 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


15:40:38 W. Europe Daylight Time Billed flex credit cost: 3.667.

...

15:40:56 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


15:41:01 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [6]:
values.keys()

dict_keys([3.4])

In [7]:

data = {
    "n_values": n_unique,
    "ff_values": ff_unique,
    "size_values": size_unique,
    "z_values": z_unique,
    "transmission_data": values,
    "lambdas": lambdas
}


create_hdf5_from_dict(data,f"{dir}/20260422_diffraction_n_3.4_ff_0.2172_ffh_0.225_schulz.h5")